In [10]:
'''today we are going to handwrite a system prompt that forces the model to output in strict 'Thought:/ Action: / Action Input: / Observation:'
format non-tool calling API, just plain text the llm generates and I parse myself.
The parser: writing a small regex/string parser that extracts the action and Action input from
the model's raw text output, since you're not using structured tool-calling here.
the loop: Building a while loop that (1) Sends the prompt, (2)parses the action, (3)executes the matching Python function,
(4)injects the result back in as observation:, (5)repeats until the model outputs Final answer'''

# importing necessary libraries
import os
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from groq import Groq

load_dotenv(find_dotenv())
client = Groq()

Settings.embed_model = HuggingFaceEmbedding(
    model_name= 'BAAI/bge-m3'
)



In [11]:
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [12]:
# function definition

def search_local_docs(query:str) -> str:
    """Searches my local Knowledge base containing historical Apple 10-K financial documents
    (covering fiscal years up to 2024). Use this to retrieve historical sales, net revenue,
    and internal corporate performance figures.
    
    CRITICAL: Do not pass comparative or converstional questions here.
    Convert queries into strict financial line items, such as:
    - 'Apple consolidated statements of operations net sales' 
    - 'Apple total net slaes 2023 -2024' 
    - 'Summary of operations data'
    """

    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)
    return "\n\n".join([doc.node.get_content() for doc in results])

def get_doc_years(_: str = "") -> str:
    """Returns the list of available years in the 10k pdf"""
    return "Available years: 2021, 2022, 2023"

def calculator(expression: str) -> str:
    """Evaluates a basic math expression and literally anything to do with math calculations
    ."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"
    
TOOLS = {
    "search_local_docs": search_local_docs,
    "get_doc_years": get_doc_years,
    "calculator": calculator,
}


In [ ]:
# it's now time for the react system prompt

REACT_SYSTEM_PROMPT = """You are a careful search assistant that solves problems step by step

You have access to these tools:
-search_local_docs(query): searches the user's local pdf documents.
-get_doc_years(): lists the number of years available in the apple 10K document.
-calculator(expression): evaluates a math expression

Use EXACTLY this format, one step at a time:

Thought: <your reasoning about what to do next>
Action: <one of: search_local_docs, get_doc_years, calculator>
Action Input: <the input to the tool>

After you receive an Observation, continue with another Thought/Action/Action Input,
or if you have enough information, respond with:

Thought: <final reasoning>
Final Answer: <your answer to the user>

Never skip the Thought step. Never output Action and Final Answer in the same turn.


"""

In [ ]:
# time to write code for the parser now

import re

def parse_action(text: str):
    """Pulls Action, Action Input and Final Answer if it's there out of the model's raw text output"""
    action_match = re.search(r"Action:\s*(\w+)", text)
    input_match = re.search(r"Action Input:\s*(.+)", text)
    final_match = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)

    if final_match:
        return None, None, final_match.group(1).strip()
    
    action = action_match.group(1).strip() if action_match else None
    action_input = input_match.group(1).strip() if input_match else None
    return action, action_input, None

In [15]:
def run_react_loop(question: str, max_steps: int = 6):
    conversation = f"{REACT_SYSTEM_PROMPT}\n\nQuestion: {question}\n"
    print(f"Question: {question}\n")

    for step in range(max_steps):
        response = client.chat.completions.create(
            model = "llama-3.3-70B-versatile",
            messages = [{"role": "user", "content": conversation}],
            temperature= 0,
            stop = ["Observation:"] # Stopping the model before it fakes an observation
        )

        text = response.choices[0].message.content
        print(f"---Step {step + 1} ---\n{text}\n")

        # Saving what the model just said to the running history
        conversation += f"{text}\n"

        action, action_input, final_answer = parse_action(text)

        if final_answer:
            print(f"👍 Final answer: {final_answer}")
            return final_answer
        
        if action and action in TOOLS:
            observation = TOOLS[action](action_input)
        elif action:
            observation = f"Error: tool '{action}' does not exist. Available tools: {list(TOOLS.keys())}"
        else:
            observation = "Error: could not parse an Action. Please follow the Thought/Action/Action Input format exactly."
        print(f"Observation: {observation}\n")

        # Feeding the tool's result back into the prompt history so the LLM can read it next turn
        conversation += f"Observation: {observation}\n"

    print("🤥 Max steps reached without Final Answer.")
    return None


In [16]:
# running it

run_react_loop("How many years are listed in my document and what are the revenues for each according to the document?")

Question: How many years are listed in my document and what are the revenues for each according to the document?

---Step 1 ---
Thought: To find the number of years listed in the document and the revenues for each year, I first need to understand what document we are referring to. Since the question mentions "your document" and we have access to a specific function called get_doc_years() that lists the number of years available in the Apple 10K document, it seems reasonable to assume the document in question is the Apple 10K document. Therefore, my first step should be to find out how many years are listed in this document.

Action: get_doc_years
Action Input: None

Observation: Available years: 2021, 2022, 2023

---Step 2 ---
Thought: Now that I know the number of years listed in the Apple 10K document, the next step is to find the revenues for each of these years. Since the question asks for revenues according to the document, I should search the local documents for the specific reve

'The revenues for 2022 and 2023 are $394,328 and $383,285, respectively. We were unable to find the revenue for 2021.'